# Qwen3 Causal LM Hyperparameter Sweep

Launches the subprocess-based sweep runner and summarizes Aim-backed results.

In [ ]:
from pathlib import Path
import json
import os
import signal
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "qwen3_0.6b_casual_lm_sweep.py").exists():
    notebook_dir = cwd
elif (cwd / "lora-fine-tuning" / "qwen3_0.6b_casual_lm_sweep.py").exists():
    notebook_dir = cwd / "lora-fine-tuning"
else:
    raise FileNotFoundError("Could not find qwen3_0.6b_casual_lm_sweep.py")

script_path = notebook_dir / "qwen3_0.6b_casual_lm_sweep.py"
results_root = notebook_dir / "results"

def terminate_process_tree(process, timeout=30):
    if process.poll() is not None:
        return
    if os.name == "nt":
        process.terminate()
    else:
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            return
        except Exception:
            process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=timeout)
        return
    except subprocess.TimeoutExpired:
        pass
    if os.name == "nt":
        process.terminate()
    else:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            return
        except Exception:
            process.terminate()
    try:
        process.wait(timeout=10)
        return
    except subprocess.TimeoutExpired:
        pass
    if os.name == "nt":
        process.kill()
    else:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            return
        except Exception:
            process.kill()
    process.wait()


def latest_sweep_dir():
    sweeps_root = results_root / "sweeps"
    sweep_dirs = sorted(
        [path for path in sweeps_root.glob("qwen3_clm_sweep_*") if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    return sweep_dirs[-1] if sweep_dirs else None


def print_failure_summary():
    sweep_dir = latest_sweep_dir()
    if sweep_dir is None:
        print("No sweep directory found yet.")
        return
    summary_path = sweep_dir / "summary.csv"
    if not summary_path.exists():
        print(f"No summary.csv found in {sweep_dir}")
        return
    summary = pd.read_csv(summary_path)
    failed = summary[summary["status"].isin(["failed", "interrupted", "metrics_json_invalid"])]
    if failed.empty:
        print(f"No failed rows in {summary_path}")
        return
    display(failed[["config_index", "status", "error_type", "error_message", "run_dir"]])


def run_command(command):
    print(" ".join(command))
    process = subprocess.Popen(
        command,
        cwd=notebook_dir,
        text=True,
        start_new_session=(os.name != "nt"),
    )
    try:
        return_code = process.wait()
    except KeyboardInterrupt:
        terminate_process_tree(process)
        raise
    if return_code != 0:
        print_failure_summary()
        raise RuntimeError(f"Command exited with code {return_code}; see failed config details above.")


print(f"Python: {sys.executable}")
print(f"Sweep script: {script_path}")
print(f"Results root: {results_root}")

## Configurations

In [ ]:
configs = json.loads(
    subprocess.check_output(
        [sys.executable, str(script_path), "list-configs", "--json"],
        cwd=notebook_dir,
        text=True,
    )
)
configs_df = pd.DataFrame(configs)
configs_df[
    [
        "index",
        "group",
        "max_seq_length",
        "learning_rate",
        "lora_r",
        "lora_alpha",
        "lora_dropout",
        "train_batch_size",
        "gradient_accumulation_steps",
        "effective_batch_size",
        "config_id",
    ]
]

## Launch Sweep

In [ ]:
SWEEP_ID = None  # Set a string to resume or name a specific sweep.
RUN_FULL_SWEEP = False
CONFIG_INDEXES = [1]
SMOKE_TEST = True

command = [sys.executable, str(script_path), "run-sweep", "--resume"]
if SWEEP_ID:
    command.extend(["--sweep-id", SWEEP_ID])
if not RUN_FULL_SWEEP:
    command.extend(["--config-index", ",".join(str(index) for index in CONFIG_INDEXES)])

if SMOKE_TEST:
    command.extend([
        "--allow-non-cuda",
        "--max-steps", "1",
        "--train-limit", "8",
        "--validation-limit", "4",
        "--test-limit", "4",
        "--cooldown-seconds", "0",
    ])
else:
    cuda_check = subprocess.run(
        [sys.executable, "-c", "import torch; raise SystemExit(0 if torch.cuda.is_available() else 1)"],
        cwd=notebook_dir,
    )
    if cuda_check.returncode != 0:
        raise RuntimeError("CUDA is not available. Keep SMOKE_TEST=True for local checks or run this on the A100 box.")

run_command(command)

## Summary

In [ ]:
sweeps_root = results_root / "sweeps"
sweep_dirs = sorted(
    [path for path in sweeps_root.glob("qwen3_clm_sweep_*") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
)
if not sweep_dirs:
    raise FileNotFoundError(f"No sweep directories found under {sweeps_root}")

SWEEP_DIR = sweep_dirs[-1]
summary_path = SWEEP_DIR / "summary.csv"
summary = pd.read_csv(summary_path)
print(f"Loaded: {summary_path}")

leaderboard = summary.sort_values(
    ["status", "validation_f1", "test_f1"],
    ascending=[True, False, False],
)
leaderboard[
    [
        "status",
        "config_index",
        "group",
        "max_seq_length",
        "learning_rate",
        "lora_r",
        "lora_alpha",
        "lora_dropout",
        "validation_f1",
        "validation_accuracy",
        "validation_precision",
        "validation_recall",
        "test_f1",
        "test_accuracy",
        "test_precision",
        "test_recall",
        "run_name",
    ]
]

## Rebuild Summary

In [ ]:
subprocess.run(
    [sys.executable, str(script_path), "summarize", "--sweep-dir", str(SWEEP_DIR)],
    cwd=notebook_dir,
    check=True,
)
summary = pd.read_csv(SWEEP_DIR / "summary.csv")
summary.sort_values(["status", "validation_f1", "test_f1"], ascending=[True, False, False])